# Vizualise correlations & cell assemblies activity

Load packages

In [ ]:
from scipy import signal
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, Cursor
from scipy import fftpack
import pandas as pd
from pathlib import Path
import os
import copy
import json
from IPython.display import display
from ipyfilechooser import FileChooser
from ephyviewer import mkQApp, MainViewer, TraceViewer
from ephyviewer import AnalogSignalSourceWithScatter
import ephyviewer
from scipy.stats import zscore
from scipy.interpolate import interp1d
from itertools import groupby
import sys 
import pickle
import seaborn as sns
from scipy.signal import find_peaks
from scipy.signal import chirp, find_peaks, peak_widths
from scipy import stats
import matplotlib.colors as mcolors
import warnings
from scipy import interpolate
import re
from scipy.stats import f_oneway
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu
from scipy.stats import ks_2samp
import diptest
import random
from collections import Counter, defaultdict
import seaborn as sns
import pandas as pd
from matplotlib.colors import ListedColormap
import networkx as nx

from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import ast
warnings.filterwarnings("ignore")

#%matplotlib widget

## Correlation across vigilance states

In [ ]:
try: # tries to retrieve dfilepath either from a previous run or from a previous notebook
    %store -r dfilepath
except:
    print("the path was not defined in store")
    #dpath = "/Users/mb/Documents/Syntuitio/AudreyHay/PlanB/ExampleRedLines/2022_08_06/13_30_01/My_V4_Miniscope/"
dfilepath = "//10.69.168.1/crnldata/waking/audrey_hay/L1imaging/"

fc1 = FileChooser(dfilepath,select_default=True, show_only_dirs = False, title = "<b>Load excel or pkl file</b>", layout=widgets.Layout(width='100%'))
display(fc1)

# Sample callback function
def update_my_folder(chooser):
    global dfilepath
    dfilepath = chooser.selected
    %store dfilepath
    return 

# Register callback function
fc1.register_callback(update_my_folder)

In [ ]:
try :
    combined_df = pd.read_excel(dfilepath, index_col=0 , sheet_name=None)
except:
    with open(dfilepath, 'rb') as pickle_file:
        combined_df = pickle.load(pickle_file)
        
# dfilepath="//10.69.168.1/crnldata/waking/audrey_hay/L1imaging/Analysed2025_AB/_global_analysis/AVG_VigSt_2025-04-17_15_22_48_TEST/Baseline/Cluster0units/L1NDNF_mice_VigSt_PairCorrCa_Cluster0units.xlsx"
everysheat=1
nbplot=int(len(combined_df)/everysheat)
fig, axes = plt.subplots(1, nbplot, figsize=(15, 5))
axes = np.atleast_1d(axes)  # Makes axes indexable even if it's just one
VMAX=1
VMIN=-.2

plotnb=0

for i in np.arange(len(combined_df))[::everysheat]:
    key = list(combined_df)[i]
        
    CaCorrMatrix=combined_df[key]
    df = CaCorrMatrix.apply(pd.to_numeric, errors='coerce')
    ax = sns.heatmap(df, ax=axes[plotnb], square=True, vmin=VMIN , vmax=VMAX , cmap='viridis',
                                    cbar_kws={'shrink': 0.6, 'aspect': 20})
    
    try:
        # Set only the first and last ticks for x and y axes
        # Get the number of ticks
        x_ticks = ax.get_xticks()
        y_ticks = ax.get_yticks()
        ax.set_xticks([x_ticks[0], x_ticks[-1]])
        ax.set_yticks([y_ticks[0], y_ticks[-1]])

        # Optionally, set the labels for these ticks if needed
        ax.set_xticklabels([f'#{int(x_ticks[0])+1}', f'#{int(x_ticks[-1])+1}'],size=6)
        ax.set_yticklabels([f'#{int(y_ticks[0])+1}', f'#{int(y_ticks[-1])+1}'],size=6)
    except: pass

    # Optionally, add labels for the axes if needed
    ax.set_xlabel('Neurons', labelpad=-2,size=8) 
    ax.set_ylabel('Neurons', labelpad=-8,size=8) 
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=6)  # Adjust the font size here
    cbar.set_label('Correlation coefficient (r)', fontsize=8, font='Arial', rotation=-90, labelpad=10)
    ax.xaxis.set_ticks_position('top')  # Move ticks to the top
    ax.xaxis.set_label_position('top')  # Move label to the top
    axes[plotnb].set_title(f'{key} (r={(np.round(np.nanmean(CaCorrMatrix),2))})', fontsize=12)
    plotnb+=1


#plt.savefig(f'C:/Users/Manip2/Documents/ElifePaper/Rawdata/Extract{mice}_{folder_base.parts[-2]}_3VigSt_{beg}to{fin}s_2heatmap.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.tight_layout()
plt.show()

## Cell assemblies activity across vigilance states

Plot Cell assembly activity relative to Vig States

In [ ]:
Analysisfolder= 'VigSt_2025-10-15_10_03_15_goodCellAssID'
Analysisfolder= 'VigSt_2025-10-24_08_58_48_CellAss_with_EstimSpike'
Analysisfolder= 'VigSt_2025-10-24_10_12_36_CellAss_with_EstimSpike_100ms'
Analysisfolder= ''


In [ ]:
dfilepath=f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/{Analysisfolder}/CellAssembly_Global.pkl"
with open(dfilepath, 'rb') as pickle_file:
    df_CellAss_origin = pickle.load(pickle_file)

drug= 'baseline'
NrSubtype='L2_3_mice' # L1NDNF_mice OR L2_3_mice

df_CellAss=df_CellAss_origin.copy()
df_CellAss = df_CellAss[df_CellAss['SubstateDuration'] != 0]
# Convert string representations of lists into actual Python lists.
df_CellAss['Cells_in_Assembly']=df_CellAss['Cells_in_Assembly'] = df_CellAss['Cells_in_Assembly'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
# Filter the DataFrame to only keep rows where the number of cells in the list matches the 'Assembly_size'.
df_CellAss = df_CellAss[df_CellAss['Cells_in_Assembly'].apply(len) == df_CellAss['Assembly_size']]

# with open(dfilepath, 'wb') as pickle_file:
#    pickle.dump(df_CellAss, pickle_file) # WHY??

#df_CellAss_= df_CellAss[df_CellAss['NeuronType']==NrSubtype]
#df_CellAss_=df_CellAss_[df_CellAss_['Drug'] == drug]


plt.figure(figsize=(6, 6))
table_CellAss = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['Substate']], values='Avg_Activity', aggfunc='mean', fill_value=None)
desired_order = ['AW','QW', 'NREM', 'REM']   
try:del table_CellAss['undefined']
except: pass
try:del table_CellAss['IS']
except: pass
try: table_CellAss = table_CellAss[desired_order]
except: pass
plt.subplot(2,2,1)
plt.plot(table_CellAss.columns, table_CellAss.values.T, alpha=0.5, linewidth=2)
plt.plot(table_CellAss.columns, table_CellAss.mean(), linewidth=2, color='black')
plt.errorbar(table_CellAss.columns, table_CellAss.mean(), yerr=table_CellAss.sem() ,
             fmt='o', color='black', capsize=5)
plt.ylabel('Avg activity per cell assemblies')
plt.title(f'{NrSubtype}, n={len(table_CellAss)}, {drug}')
plt.tight_layout()

groups = [table_CellAss[col].dropna().values for col in table_CellAss.columns]
f_stat, p_value = f_oneway(*groups)
print(f"ANOVA result: F = {f_stat:.3f}, p = {p_value:.3e}")
df_melt = table_CellAss.melt(var_name='Substates', value_name='Avg_Activity')
df_melt = df_melt.dropna(subset=['Avg_Activity', 'Substates'])
df_melt['Avg_Activity'] = pd.to_numeric(df_melt['Avg_Activity'], errors='coerce')
tukey = pairwise_tukeyhsd(endog=df_melt['Avg_Activity'], groups=df_melt['Substates'], alpha=0.05)
print(tukey.summary())
print(len((df_melt)))

try:
    table_CellAss = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['Substate']], values='EventFreq', aggfunc='mean', fill_value=None)
    try:del table_CellAss['undefined']
    except: pass
    try:del table_CellAss['IS']
    except: pass
    try: table_CellAss = table_CellAss[desired_order]
    except: pass
    plt.subplot(2,2,3)
    plt.plot(table_CellAss.columns, table_CellAss.values.T, alpha=0.5, linewidth=2)
    plt.plot(table_CellAss.columns, table_CellAss.mean(), linewidth=2, color='black')
    plt.errorbar(table_CellAss.columns, table_CellAss.mean(), yerr=table_CellAss.sem() ,
                fmt='o', color='black', capsize=5)
    plt.ylabel('Event frequency per cell assemblies')

    groups = [table_CellAss[col].dropna().values for col in table_CellAss.columns]
    f_stat, p_value = f_oneway(*groups)
    print(f"ANOVA result: F = {f_stat:.3f}, p = {p_value:.3e}")
    df_melt = table_CellAss.melt(var_name='Substates', value_name='EventFreq')
    df_melt = df_melt.dropna(subset=['Avg_Activity', 'Substates'])
    df_melt['EventFreq'] = pd.to_numeric(df_melt['EventFreq'], errors='coerce')
    tukey = pairwise_tukeyhsd(endog=df_melt['EventFreq'], groups=df_melt['Substates'], alpha=0.05)
    print(tukey.summary())
    print(len((df_melt)))
except: pass

plt.savefig(f'{Path(dfilepath).parent}/CellAssembly_VigStActivity_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

## Place cells, HD cells & Cell assembly

Load NeuronIdentity file & assign Neuron ID to each neuron

In [ ]:
with open("//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/NeuronIdentity_2025-10-10_11_13_40/List_SignCells.pkl", 'rb') as pickle_file:
    df_MazeCellID = pickle.load(pickle_file)

AllCheeseboard_cells=np.array(df_MazeCellID['all'])
HD_cells=np.array(df_MazeCellID['HD'])
PC_cells=np.array(df_MazeCellID['PC'])
PCHD_cells=np.array(df_MazeCellID['PC&HD'])

Spatial_cells=np.concatenate([PCHD_cells,  HD_cells, PC_cells])

dfilepathV=f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/{Analysisfolder}/VigStates_Global.pkl"
with open(dfilepathV, 'rb') as pickle_file:
    df_VigSt = pickle.load(pickle_file)

All_cells= np.array(df_VigSt['Unit_ID'].unique().tolist())

id_to_cluster = {
    p: (
        'HD' if p in HD_cells.tolist()
        else 'PC' if p in PC_cells.tolist()
        else 'PC&HD' if p in PCHD_cells.tolist()
        else 'notPCnotHD' if p in AllCheeseboard_cells.tolist() and p not in (HD_cells.tolist() + PC_cells.tolist() + PCHD_cells.tolist())
        else 'unclassified'
    )
    for p in All_cells.tolist()
}
notPCnotHD_cells = np.array([name for name, classification in id_to_cluster.items() if classification == 'notPCnotHD'])
unclassified_cells = np.array([name for name, classification in id_to_cluster.items() if classification == 'unclassified'])


Spatial_id_to_cluster = {
    p: (
        'Spatial' if p in Spatial_cells.tolist()
        else 'Non_Spatial' if p in AllCheeseboard_cells.tolist() and p not in (HD_cells.tolist() + PC_cells.tolist() + PCHD_cells.tolist())
        else 'unclassified'
    )
    for p in All_cells.tolist()
}

Non_Spatial_cells = np.array([name for name, classification in Spatial_id_to_cluster.items() if classification == 'Non_Spatial'])

Save Vigilance State with Neuron ID

In [ ]:
df_VigSt['Neuron_Identity'] = df_VigSt['Unit_ID'].map(id_to_cluster)
df_VigSt['Spatially_tuned'] = df_VigSt['Unit_ID'].map(Spatial_id_to_cluster)

filenameOut = Path(dfilepathV).parent / f'VigStates_Global_NeuronID.pkl'
with open(filenameOut, 'wb') as pickle_file:
    pickle.dump(df_VigSt, pickle_file)
    
filenameOut = Path(dfilepathV).parent / f'VigStates_Global_NeuronID.xlsx'
with pd.ExcelWriter(filenameOut) as writer:
    df_VigSt.to_excel(writer, index=False)

Identify Cell assembly Types and test proportion

In [ ]:
df_CellAss_uniqAss = df_CellAss.drop_duplicates(subset='Assembly_ID', keep='first')
df_CellAss_uniqAss = df_CellAss_uniqAss[df_CellAss_uniqAss['Cells_in_Assembly'].apply(len) == df_CellAss_uniqAss['Assembly_size']]
groups = df_CellAss_uniqAss['Cells_in_Assembly'].tolist()
ids=All_cells.tolist()
cluster_labels = ['HD']*len(HD_cells) + ['PC']*len(PC_cells) + ['PC&HD']*len(PCHD_cells) + ['notPCnotHD']*len(notPCnotHD_cells) + ['unclassified']*len(unclassified_cells)

def classify_groups(groups, id_to_cluster):
    labels = []
    for group in groups:
        clusters = [id_to_cluster.get(i, np.nan) for i in group]
        if len(clusters) >= 1:
            count = Counter(clusters)
            top, n_top = count.most_common(1)[0]
            top_clusters = [k for k, v in count.items() if v == n_top]        
            # If there are multiple top clusters or the majority is less than 50%, label as "mix"
            if len(top_clusters) == 1 and (n_top / len(group) * 100) > 50 and not pd.isna(top):
                labels.append(top)
            else:
                labels.append("mix")  
        else:
            labels.append("mix")    
    return labels

# --- Observed classification
observed_labels = classify_groups(groups, id_to_cluster)
observed_counts = Counter(observed_labels)
df_CellAss_uniqAss['Assembly_Neurons_ID']= observed_labels

# --- Permutation test
n_perms = 5000
null_dists = defaultdict(list)

for _ in range(n_perms):
    # Shuffle cluster labels across individuals
    shuffled_clusters = dict(zip(ids, random.sample(cluster_labels, len(cluster_labels))))
    shuffled_labels = classify_groups(groups, shuffled_clusters)
    counts = Counter(shuffled_labels)
    for label in observed_counts:
        null_dists[label].append(counts.get(label, 0))

# --- Z-scores and p-values
results = []
for label in observed_counts:
    obs = observed_counts[label]
    null = null_dists[label]
    mean = np.mean(null)
    std = np.std(null)
    z = (obs - mean) / std if std > 0 else 0
    # Two-tailed p-value
    p = (np.sum(np.abs(np.array(null) - mean) >= abs(obs - mean)) + 1) / (n_perms + 1)
    results.append((label, obs, mean, std, z, p))

# --- Output
print("\nAssembly Neuron ID Distribution Test")
print(f"{'ID':>10} {'Obs':>5} {'Mean':>7} {'Std':>7} {'Z':>6} {'p-value':>8}")
for label, obs, mean, std, z, p in sorted(results, key=lambda x: x[-1]):
    print(f"{label:>10} {obs:5} {mean:7.2f} {std:7.2f} {z:6.2f} {p:8.4f}")


df_perm = pd.DataFrame(results, columns=["Cell_Assembly_ID", "Obs", "PermMean", "Std", "Z", "p"])
df_perm['Obs_prop'] = df_perm['Obs'] / df_perm['Obs'].sum() *100
df_perm['Perm_prop'] = df_perm['PermMean'] / df_perm['PermMean'].sum() *100
df_perm["Perm_std"] = df_perm["Std"] / df_perm["PermMean"].sum() *100

df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "unclassified", "mix"])
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "unclassified", "mix"])
fig, ax= plt.subplots(figsize=(3, 3))
color_map = ['blue',  'green', 'orange','black', 'grey', 'purple']
for x, yperm, yobs, err, color in zip(df_perm["Cell_Assembly_ID"], df_perm["Perm_prop"], df_perm["Obs_prop"], df_perm["Perm_std"], color_map):
    line1 = ax.errorbar(x, yperm, yerr=err, fmt='o', color=color, capsize=5, alpha = 0.5, label = 'Permuted ± std')
    line2 = ax.scatter(x, yobs, color=color, label = 'Observed')
    ax.legend([line2, line1], ['Observed', 'Permuted ± std']) if x == 'unclassified' else None
plt.ylabel("Cell assemblies (%)")
plt.ylim(0, 100)
plt.title(f"Cell Assembly Types (n={len(df_CellAss_uniqAss)})")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssembly_Proportion_permvsreal_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
df_SpCellAss_uniqAss = df_CellAss.drop_duplicates(subset='Assembly_ID', keep='first')
df_SpCellAss_uniqAss = df_SpCellAss_uniqAss[df_SpCellAss_uniqAss['Cells_in_Assembly'].apply(len) == df_SpCellAss_uniqAss['Assembly_size']]
groups = df_SpCellAss_uniqAss['Cells_in_Assembly'].tolist()
ids=All_cells.tolist()
cluster_labels = ['Spatial']*len(Spatial_cells) + ['Non_Spatial']*len(Non_Spatial_cells) + ['unclassified']*len(unclassified_cells)

def classify_groups(groups, Spatial_id_to_cluster):
    labels = []
    for group in groups:
        clusters = [Spatial_id_to_cluster.get(i, np.nan) for i in group]
        if len(clusters) >= 1:
            count = Counter(clusters)
            top, n_top = count.most_common(1)[0]
            top_clusters = [k for k, v in count.items() if v == n_top]        
            # If there are multiple top clusters or the majority is less than 50%, label as "mix"
            if len(top_clusters) == 1 and (n_top / len(group) * 100) > 50 and not pd.isna(top):
                labels.append(top)
            else:
                labels.append("mix")  
        else:
            labels.append("mix")    
    return labels

# --- Observed classification
observed_labels = classify_groups(groups, Spatial_id_to_cluster)
observed_counts = Counter(observed_labels)
df_SpCellAss_uniqAss['Assembly_Spatial_ID']= observed_labels

# --- Permutation test
n_perms = 5000
null_dists = defaultdict(list)

for _ in range(n_perms):
    # Shuffle cluster labels across individuals
    shuffled_clusters = dict(zip(ids, random.sample(cluster_labels, len(cluster_labels))))
    shuffled_labels = classify_groups(groups, shuffled_clusters)
    counts = Counter(shuffled_labels)
    for label in observed_counts:
        null_dists[label].append(counts.get(label, 0))

# --- Z-scores and p-values
results = []
for label in observed_counts:
    obs = observed_counts[label]
    null = null_dists[label]
    mean = np.mean(null)
    std = np.std(null)
    z = (obs - mean) / std if std > 0 else 0
    # Two-tailed p-value
    p = (np.sum(np.abs(np.array(null) - mean) >= abs(obs - mean)) + 1) / (n_perms + 1)
    results.append((label, obs, mean, std, z, p))

# --- Output
print("\nAssembly Neuron ID Distribution Test")
print(f"{'ID':>10} {'Obs':>5} {'Mean':>7} {'Std':>7} {'Z':>6} {'p-value':>8}")
for label, obs, mean, std, z, p in sorted(results, key=lambda x: x[-1]):
    print(f"{label:>10} {obs:5} {mean:7.2f} {std:7.2f} {z:6.2f} {p:8.4f}")


df_perm = pd.DataFrame(results, columns=["Cell_Assembly_ID", "Obs", "PermMean", "Std", "Z", "p"])
df_perm['Obs_prop'] = df_perm['Obs'] / df_perm['Obs'].sum() *100
df_perm['Perm_prop'] = df_perm['PermMean'] / df_perm['PermMean'].sum() *100
df_perm["Perm_std"] = df_perm["Std"] / df_perm["PermMean"].sum() *100
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["Spatial", "Non_Spatial", "unclassified", "mix"])
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["Spatial", "Non_Spatial", "unclassified", "mix"])

fig, ax= plt.subplots(figsize=(3, 3))
color_map = ['blue', 'orange', 'grey', 'purple']
for x, yperm, yobs, err, color in zip(df_perm["Cell_Assembly_ID"], df_perm["Perm_prop"], df_perm["Obs_prop"], df_perm["Perm_std"], color_map):
    line1= ax.errorbar(x, yperm, yerr=err, fmt='o', color=color, capsize=5, alpha = 0.5, label = 'Permuted ± std')
    line2= ax.scatter(x, yobs, color=color, label = 'Observed')
    ax.legend([line2, line1], ['Observed', 'Permuted ± std']) if x == 'unclassified' else None
plt.ylabel("Cell assemblies (%)")
plt.ylim(0, 100)
plt.title(f"Cell Assembly Types (n={len(df_SpCellAss_uniqAss)})")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/SpatialCellAssembly_Proportion_permvsreal_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Add & save Assembly main identity

In [ ]:
df_CellAss_ID_ = pd.merge(df_CellAss,df_CellAss_uniqAss[['Assembly_ID', 'Assembly_Neurons_ID']], on='Assembly_ID', how='outer')
df_CellAss_ID = pd.merge(df_CellAss_ID_,df_SpCellAss_uniqAss[['Assembly_ID', 'Assembly_Spatial_ID']], on='Assembly_ID', how='outer')

filenameOut = Path(dfilepath).parent / f'CellAssembly_Global_ID.pkl'
with open(filenameOut, 'wb') as pickle_file:
    pickle.dump(df_CellAss_ID, pickle_file)    
filenameOut = Path(dfilepath).parent / f'CellAssembly_Global_ID.xlsx'
with pd.ExcelWriter(filenameOut) as writer:
    df_CellAss_ID.to_excel(writer, index=False)

Gather similar cell assemblies (> 50% neurons in common)

In [ ]:
df_CellAss_ID_uniqAss = df_CellAss_ID.drop_duplicates(subset='Assembly_ID', keep='first')
cells_list = df_CellAss_ID_uniqAss['Cells_in_Assembly'].tolist()
assembly_ids = df_CellAss_ID_uniqAss['Assembly_ID'].tolist()
assembly_sizes = df_CellAss_ID_uniqAss['Assembly_size'].tolist()  
n_rows = len(cells_list)

def more_than_half_common(list1, list2):
    set1 = set(list1)
    set2 = set(list2)
    n_common = len(set1 & set2)
    return n_common / len(set2) > 0.5

# Compute matched assemblies
matched_assemblies = []
for i in range(n_rows):
    matches = []
    for j in range(n_rows):
        if i != j and more_than_half_common(cells_list[j], cells_list[i]):
            matches.append(assembly_ids[j])
    matched_assemblies.append(matches)
df_CellAss_ID_uniqAss['Matching_Assemblies'] = matched_assemblies

# Compute New_Assembly_ID using the largest Assembly_size
def largest_match(row):
    if not row['Matching_Assemblies']:
        return row['Assembly_ID']
    else:
        # Get sizes of all matched assemblies
        sizes = [assembly_sizes[assembly_ids.index(aid)] for aid in row['Matching_Assemblies']]
        # Find index of max size
        max_idx = sizes.index(max(sizes))
        return row['Matching_Assemblies'][max_idx]
df_CellAss_ID_uniqAss['New_Assembly_ID'] = df_CellAss_ID_uniqAss.apply(largest_match, axis=1)

while True: # If a cell assembly B replacing a cell assembly A is being replaced by a cell assembly C, then cell assembly A become C
    m = df_CellAss_ID_uniqAss.set_index('Assembly_ID')['New_Assembly_ID']
    new_B = df_CellAss_ID_uniqAss['New_Assembly_ID'].map(m).fillna(df_CellAss_ID_uniqAss['New_Assembly_ID'])
    if new_B.equals(df_CellAss_ID_uniqAss['New_Assembly_ID']): break
    df_CellAss_ID_uniqAss['New_Assembly_ID'] = new_B

df_NewCellAss = pd.merge(df_CellAss_ID,df_CellAss_ID_uniqAss[['Assembly_ID', 'New_Assembly_ID']], on='Assembly_ID', how='outer')

df_NewCellAss.loc[df_NewCellAss['Assembly_ID'] != df_NewCellAss['New_Assembly_ID'], 'Assembly_Neurons_ID'] = np.nan 
df_NewCellAss.loc[df_NewCellAss['Assembly_ID'] != df_NewCellAss['New_Assembly_ID'], 'Assembly_Spatial_ID'] = np.nan 
df_NewCellAss.loc[df_NewCellAss['Assembly_ID'] != df_NewCellAss['New_Assembly_ID'], 'Assembly_size'] = np.nan 
df_NewCellAss.loc[df_NewCellAss['Assembly_ID'] != df_NewCellAss['New_Assembly_ID'], 'Cells_in_Assembly'] = np.nan 

Identify New Cell assembly Types and test proportion

In [ ]:
df_NewCellAss_uniqAss= df_NewCellAss[df_NewCellAss['Assembly_Neurons_ID'].notna()]
df_NewCellAss_uniqAss = df_NewCellAss_uniqAss.drop_duplicates(subset='New_Assembly_ID', keep='first')
groups = df_NewCellAss_uniqAss['Cells_in_Assembly'].tolist()
ids=All_cells.tolist()
cluster_labels = ['HD']*len(HD_cells) + ['PC']*len(PC_cells) + ['PC&HD']*len(PCHD_cells) + ['notPCnotHD']*len(notPCnotHD_cells) + ['unclassified']*len(unclassified_cells)

def classify_groups(groups, id_to_cluster):
    labels = []
    for group in groups:
        clusters = [id_to_cluster.get(i, np.nan) for i in group]
        if len(clusters) >= 1:
            count = Counter(clusters)
            top, n_top = count.most_common(1)[0]
            top_clusters = [k for k, v in count.items() if v == n_top]        
            # If there are multiple top clusters or the majority is less than 50%, label as "mix"
            if len(top_clusters) == 1 and (n_top / len(group) * 100) > 50 and not pd.isna(top):
                labels.append(top)
            else:
                labels.append("mix")  
        else:
            labels.append("mix")    
    return labels

# --- Observed classification
observed_labels = classify_groups(groups, id_to_cluster)
observed_counts = Counter(observed_labels)
df_NewCellAss_uniqAss['Assembly_Neurons_ID']= observed_labels

# --- Permutation test
n_perms = 5000
null_dists = defaultdict(list)

for _ in range(n_perms):
    # Shuffle cluster labels across individuals
    shuffled_clusters = dict(zip(ids, random.sample(cluster_labels, len(cluster_labels))))
    shuffled_labels = classify_groups(groups, shuffled_clusters)
    counts = Counter(shuffled_labels)
    for label in observed_counts:
        null_dists[label].append(counts.get(label, 0))

# --- Z-scores and p-values
results = []
for label in observed_counts:
    obs = observed_counts[label]
    null = null_dists[label]
    mean = np.mean(null)
    std = np.std(null)
    z = (obs - mean) / std if std > 0 else 0
    # Two-tailed p-value
    p = (np.sum(np.abs(np.array(null) - mean) >= abs(obs - mean)) + 1) / (n_perms + 1)
    results.append((label, obs, mean, std, z, p))

# --- Output
print("\nAssembly Neuron ID Distribution Test")
print(f"{'ID':>10} {'Obs':>5} {'Mean':>7} {'Std':>7} {'Z':>6} {'p-value':>8}")
for label, obs, mean, std, z, p in sorted(results, key=lambda x: x[-1]):
    print(f"{label:>10} {obs:5} {mean:7.2f} {std:7.2f} {z:6.2f} {p:8.4f}")


df_perm = pd.DataFrame(results, columns=["Cell_Assembly_ID", "Obs", "PermMean", "Std", "Z", "p"])
df_perm['Obs_prop'] = df_perm['Obs'] / df_perm['Obs'].sum() *100
df_perm['Perm_prop'] = df_perm['PermMean'] / df_perm['PermMean'].sum() *100
df_perm["Perm_std"] = df_perm["Std"] / df_perm["PermMean"].sum() *100

df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "unclassified", "mix"])
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "unclassified", "mix"])

fig, ax= plt.subplots(figsize=(3, 3))
color_map = ['blue', 'green', 'orange', 'black', 'grey', 'purple']
for x, yperm, yobs, err, color in zip(df_perm["Cell_Assembly_ID"], df_perm["Perm_prop"], df_perm["Obs_prop"], df_perm["Perm_std"], color_map):
    line1= ax.errorbar(x, yperm, yerr=err, fmt='o', color=color, capsize=5, alpha = 0.5, label = 'Permuted ± std')
    line2= ax.scatter(x, yobs, color=color, label = 'Observed')
    ax.legend([line2, line1], ['Observed', 'Permuted ± std']) if x == 'unclassified' else None
plt.ylabel("Cell assemblies (%)")
plt.ylim(0, 100)
plt.title(f"Cell Assembly Types (n={len(df_NewCellAss_uniqAss)})")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssembly_Proportion_permvsreal_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
df_SpNewCellAss_uniqAss= df_NewCellAss[df_NewCellAss['Assembly_Neurons_ID'].notna()]
df_SpNewCellAss_uniqAss = df_SpNewCellAss_uniqAss.drop_duplicates(subset='New_Assembly_ID', keep='first')
groups = df_SpNewCellAss_uniqAss['Cells_in_Assembly'].tolist()
ids=All_cells.tolist()
cluster_labels = ['Spatial']*len(Spatial_cells) + ['Non_Spatial']*len(Non_Spatial_cells) + ['unclassified']*len(unclassified_cells)

def classify_groups(groups, Spatial_id_to_cluster):
    labels = []
    for group in groups:
        clusters = [Spatial_id_to_cluster.get(i, np.nan) for i in group]
        if len(clusters) >= 1:
            count = Counter(clusters)
            top, n_top = count.most_common(1)[0]
            top_clusters = [k for k, v in count.items() if v == n_top]        
            # If there are multiple top clusters or the majority is less than 50%, label as "mix"
            if len(top_clusters) == 1 and (n_top / len(group) * 100) > 50 and not pd.isna(top):
                labels.append(top)
            else:
                labels.append("mix")  
        else:
            labels.append("mix")    
    return labels

# --- Observed classification
observed_labels = classify_groups(groups, Spatial_id_to_cluster)
observed_counts = Counter(observed_labels)
df_SpNewCellAss_uniqAss['Assembly_Spatial_ID']= observed_labels

# --- Permutation test
n_perms = 5000
null_dists = defaultdict(list)

for _ in range(n_perms):
    # Shuffle cluster labels across individuals
    shuffled_clusters = dict(zip(ids, random.sample(cluster_labels, len(cluster_labels))))
    shuffled_labels = classify_groups(groups, shuffled_clusters)
    counts = Counter(shuffled_labels)
    for label in observed_counts:
        null_dists[label].append(counts.get(label, 0))

# --- Z-scores and p-values
results = []
for label in observed_counts:
    obs = observed_counts[label]
    null = null_dists[label]
    mean = np.mean(null)
    std = np.std(null)
    z = (obs - mean) / std if std > 0 else 0
    # Two-tailed p-value
    p = (np.sum(np.abs(np.array(null) - mean) >= abs(obs - mean)) + 1) / (n_perms + 1)
    results.append((label, obs, mean, std, z, p))

# --- Output
print("\nAssembly Neuron ID Distribution Test")
print(f"{'ID':>10} {'Obs':>5} {'Mean':>7} {'Std':>7} {'Z':>6} {'p-value':>8}")
for label, obs, mean, std, z, p in sorted(results, key=lambda x: x[-1]):
    print(f"{label:>10} {obs:5} {mean:7.2f} {std:7.2f} {z:6.2f} {p:8.4f}")


df_perm = pd.DataFrame(results, columns=["Cell_Assembly_ID", "Obs", "PermMean", "Std", "Z", "p"])
df_perm['Obs_prop'] = df_perm['Obs'] / df_perm['Obs'].sum() *100
df_perm['Perm_prop'] = df_perm['PermMean'] / df_perm['PermMean'].sum() *100
df_perm["Perm_std"] = df_perm["Std"] / df_perm["PermMean"].sum() *100

df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["Spatial", "Non_Spatial", "unclassified", "mix"])
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["Spatial", "Non_Spatial", "unclassified", "mix"])

fig, ax= plt.subplots(figsize=(3, 3))
color_map = ['blue', 'orange', 'grey', 'purple']
for x, yperm, yobs, err, color in zip(df_perm["Cell_Assembly_ID"], df_perm["Perm_prop"], df_perm["Obs_prop"], df_perm["Perm_std"], color_map):
    line1= ax.errorbar(x, yperm, yerr=err, fmt='o', color=color, capsize=5, alpha = 0.5, label = 'Permuted ± std')
    line2= ax.scatter(x, yobs, color=color, label = 'Observed')
    ax.legend([line2, line1], ['Observed', 'Permuted ± std']) if x == 'unclassified' else None
plt.ylabel("Cell assemblies (%)")
plt.ylim(0, 100)
plt.title(f"Cell Assembly Types (n={len(df_SpNewCellAss_uniqAss)})")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/NewSpatialCellAssembly_Proportion_permvsreal_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Save New Cell Assembly with ID

In [ ]:
df_NewCellAss=df_NewCellAss.drop('Assembly_Neurons_ID', axis =1)
df_NewCellAss=df_NewCellAss.drop('Assembly_Spatial_ID', axis =1)
df_NewCellAss=df_NewCellAss.drop('Assembly_size', axis =1)
df_NewCellAss=df_NewCellAss.drop('Cells_in_Assembly', axis =1)
df_NewCellAss=df_NewCellAss.drop('Assembly_ID', axis =1)
df_NewCellAss_ID_ = pd.merge(df_NewCellAss,df_NewCellAss_uniqAss[['New_Assembly_ID', 'Assembly_Neurons_ID','Assembly_size','Cells_in_Assembly']], on='New_Assembly_ID', how='outer')
df_NewCellAss_ID = pd.merge(df_NewCellAss_ID_,df_NewCellAss_uniqAss[['New_Assembly_ID', 'Assembly_Spatial_ID']], on='New_Assembly_ID', how='outer')

filenameOut = Path(dfilepath).parent / f'NewCellAssembly_Global_ID.pkl'
with open(filenameOut, 'wb') as pickle_file:
    pickle.dump(df_NewCellAss_ID, pickle_file)    
filenameOut = Path(dfilepath).parent / f'NewCellAssembly_Global_ID.xlsx'
with pd.ExcelWriter(filenameOut) as writer:
    df_NewCellAss_ID.to_excel(writer, index=False)

### HD vs PC vs HDPC vs notHDnotPC

Plot Reactivation of cell assemblies

In [ ]:
# Average time spent in each session types
# df_NewCellAss_ID.pivot_table(index='New_Assembly_ID', columns=['ExpeType'], values='SubstateDuration', aggfunc='sum', fill_value=None).mean()

NewCellAss_ID_pivot= df_NewCellAss_ID.pivot_table(index='New_Assembly_ID', columns=['ExpeType'], values='EventFreq', aggfunc='mean', fill_value=None)
NewCellAss_ID_pivot_clean = NewCellAss_ID_pivot.dropna(subset=['Cheeseboard', 'SleepAfter'])
NewCellAss_ID_pivot_clean1 = pd.merge(NewCellAss_ID_pivot_clean,df_NewCellAss_ID[['Assembly_Neurons_ID', 'New_Assembly_ID']], on='New_Assembly_ID', how='inner')
NewCellAss_ID_pivot_clean1 = NewCellAss_ID_pivot_clean1.drop_duplicates()

plt.figure(figsize=(2,4))
color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 'notPCnotHD': 'black', 'unclassified': 'grey', 'mix': 'purple'} 
timepoints = [1, 2]
for idx, row in NewCellAss_ID_pivot_clean1.iterrows():
    plt.plot(timepoints, [row['Cheeseboard'], row['SleepAfter']], color=color_map[row['Assembly_Neurons_ID']])
plt.xticks(timepoints, ['Cheeseboard', 'PostSession'])
plt.ylabel('Cell assembly events (Hz)')
plt.title(f'n = {len(NewCellAss_ID_pivot_clean1)}/{len(NewCellAss_ID_pivot)} cell assemblies')
plt.legend(handles=[plt.Line2D([0], [0], color=color, lw=2, label=label) for label, color in color_map.items()], title='Majority of', bbox_to_anchor=(1.05, 1), loc='upper left')
print(f'{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot)*100)} % of cell assemblies are detected during both Cheeseboard and SleepAfter sessions')
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['Cheeseboard']))*100)} % of cell assemblies detected during Cheeseboard are detected in SleepAfter sessions")
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['SleepAfter']))*100)} % of cell assemblies detected during SleepAfter were detected in Cheeseboard sessions")
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblyEvents_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
df_NewCellAss_ID_ = df_NewCellAss_ID[~(df_NewCellAss_ID['ExpeType'] == 'SleepAfter') | (df_NewCellAss_ID['Substate'] == 'NREM')]
NewCellAss_ID_pivot= df_NewCellAss_ID_.pivot_table(index='New_Assembly_ID', columns=['Substate'], values='EventFreq', aggfunc='mean', fill_value=None)
NewCellAss_ID_pivot_clean = NewCellAss_ID_pivot.dropna(subset=['AW', 'NREM'])
NewCellAss_ID_pivot_clean1 = pd.merge(NewCellAss_ID_pivot_clean,df_NewCellAss_ID_[['Assembly_Neurons_ID', 'New_Assembly_ID']], on='New_Assembly_ID', how='inner')
NewCellAss_ID_pivot_clean1 = NewCellAss_ID_pivot_clean1.drop_duplicates()

plt.figure(figsize=(2,4))
color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 'notPCnotHD': 'black', 'unclassified': 'grey', 'mix': 'purple'} 
timepoints = [1, 2]
for idx, row in NewCellAss_ID_pivot_clean1.iterrows():
    plt.plot(timepoints, [row['AW'], row['NREM']], color=color_map[row['Assembly_Neurons_ID']])
plt.xticks(timepoints, ['Cheeseboard', 'NREM PostSession'])
plt.ylabel('Cell assembly events (Hz)')
plt.title(f'n = {len(NewCellAss_ID_pivot_clean1)}/{len(NewCellAss_ID_pivot)} cell assemblies')
plt.legend(handles=[plt.Line2D([0], [0], color=color, lw=2, label=label) for label, color in color_map.items()], title='Majority of', bbox_to_anchor=(1.05, 1), loc='upper left')
print(f'{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot)*100)} % of cell assemblies are detected during both AW Cheeseboard and NREM SleepAfter sessions')
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['AW']))*100)} % of cell assemblies detected during AW Cheeseboard are detected in NREM SleepAfter sessions")
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['NREM']))*100)} % of cell assemblies detected during NREM SleepAfter were detected in AW Cheeseboard sessions")
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblyEvents_inNREM.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
df_NewCellAss_ID_ = df_NewCellAss_ID[~(df_NewCellAss_ID['ExpeType'] == 'SleepAfter') | (df_NewCellAss_ID['Substate'] == 'REM')]
NewCellAss_ID_pivot= df_NewCellAss_ID_.pivot_table(index='New_Assembly_ID', columns=['Substate'], values='EventFreq', aggfunc='mean', fill_value=None)
NewCellAss_ID_pivot_clean = NewCellAss_ID_pivot.dropna(subset=['AW', 'REM'])
NewCellAss_ID_pivot_clean1 = pd.merge(NewCellAss_ID_pivot_clean,df_NewCellAss_ID_[['Assembly_Neurons_ID', 'New_Assembly_ID']], on='New_Assembly_ID', how='inner')
NewCellAss_ID_pivot_clean1 = NewCellAss_ID_pivot_clean1.drop_duplicates()

plt.figure(figsize=(2,4))
color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 'notPCnotHD': 'black', 'unclassified': 'grey', 'mix': 'purple'} 
timepoints = [1, 2]
for idx, row in NewCellAss_ID_pivot_clean1.iterrows():
    plt.plot(timepoints, [row['AW'], row['REM']], color=color_map[row['Assembly_Neurons_ID']])
plt.xticks(timepoints, ['Cheeseboard', 'REM PostSession'])
plt.ylabel('Cell assembly events (Hz)')
plt.title(f'n = {len(NewCellAss_ID_pivot_clean1)}/{len(NewCellAss_ID_pivot)} cell assemblies')
plt.legend(handles=[plt.Line2D([0], [0], color=color, lw=2, label=label) for label, color in color_map.items()], title='Majority of', bbox_to_anchor=(1.05, 1), loc='upper left')
print(f'{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot)*100)} % of cell assemblies are detected during both AW Cheeseboard and REM SleepAfter sessions')
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['AW']))*100)} % of cell assemblies detected during AW Cheeseboard are detected in REM SleepAfter sessions")
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['REM']))*100)} % of cell assemblies detected during REM SleepAfter were detected in AW Cheeseboard sessions")
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblyEvents_inREM.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Cell assembly size

In [ ]:
NewCellAss_ID_pivot= df_NewCellAss_ID.pivot_table(index='New_Assembly_ID', columns=['ExpeType'], values='Assembly_size', aggfunc='mean', fill_value=None)
NewCellAss_ID_pivot_clean = pd.merge(NewCellAss_ID_pivot,df_NewCellAss_ID[['Assembly_Neurons_ID', 'New_Assembly_ID']], on='New_Assembly_ID', how='inner')
NewCellAss_ID_pivot_clean = NewCellAss_ID_pivot_clean.drop_duplicates()

color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 
             'notPCnotHD': 'black', 'unclassified': 'grey', 'mix': 'purple'}

df_melted = NewCellAss_ID_pivot_clean.melt(
    id_vars='Assembly_Neurons_ID',
    value_vars=['Cheeseboard', 'SleepAfter'],
    var_name='ExpeType',
    value_name='Assembly_size'
)
df_melted['Assembly_Neurons_ID'] = pd.Categorical(df_melted['Assembly_Neurons_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "unclassified", "mix"])

fig, ax= plt.subplots(figsize=(13,2.5))
sns.swarmplot(
    x='ExpeType', 
    y='Assembly_size', 
    data=df_melted, 
    hue='Assembly_Neurons_ID', 
    palette=color_map, 
    size=6, 
    alpha=1,
    ax= ax,
)
plt.ylabel('Cell Assembly size')
plt.xticks([0,1], ['Cheeseboard', 'SleepAfter'])
plt.legend(loc='upper right', title='Majority of')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblySize_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
print(f"{df_NewCellAss_ID[df_NewCellAss_ID['ExpeType']=='Cheeseboard']['New_Assembly_ID'].nunique()} cell assemblies detected during Cheeseboard sessions")
print(f"{df_NewCellAss_ID[df_NewCellAss_ID['ExpeType']=='SleepAfter']['New_Assembly_ID'].nunique()} cell assemblies detected during SleepAfter sessions")
plt.show()


In [ ]:
df_melted_ = df_melted.dropna().groupby(['ExpeType','Assembly_Neurons_ID']).size().reset_index(name='count')
pivot_df = df_melted_.pivot(index='ExpeType', columns='Assembly_Neurons_ID', values='count').fillna(0)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(7, 3), sharey=False)

color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 
             'notPCnotHD': 'black', 'unclassified': 'grey', 'mix': 'purple'}

pivot_df.plot(kind='bar', stacked=True, color=[color_map[col] for col in pivot_df.columns], ax= ax1)
ax1.set_ylabel('# of cell assemblies')
ax1.legend(title='Majority of', bbox_to_anchor=(1.05, 1), loc='upper left')
for label in ax1.get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')
ax1.set_xlabel('')

Sum = pivot_df[["PC", "HD", "PC&HD", "notPCnotHD", "unclassified", "mix"]].sum(axis=1)
pivot_df[["PC", "HD", "PC&HD", "notPCnotHD", "unclassified", "mix"]] = pivot_df[["PC", "HD", "PC&HD", "notPCnotHD", "unclassified", "mix"]].div(Sum, axis=0)*100
pivot_df.plot(kind='bar', stacked=True, color=[color_map[col] for col in pivot_df.columns], ax= ax2, legend=False)
ax2.set_ylabel('Cell assemblies (%)')
for label in ax2.get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')
ax2.set_xlabel('')

pivot_= NewCellAss_ID_pivot_clean.dropna().groupby(['Assembly_Neurons_ID']).size().reset_index(name='count')
Sum = pivot_['count'].sum(axis=0)
pivot_['count'] = pivot_['count'].div(Sum, axis=0)*100
pivot_ = pivot_.set_index('Assembly_Neurons_ID').loc[["HD", "notPCnotHD", "mix"]].reset_index()
bottom = 0
for color, value in zip(pivot_['Assembly_Neurons_ID'], pivot_['count']):
    ax3.bar(['Both sessions'], value, bottom=bottom, color=color_map[color])
    bottom += value
ax3.set_ylabel('Cell assemblies (%)')
for label in ax3.get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblyProportion_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(2.5,2.5))
df_melted_ = df_melted.dropna().groupby(['ExpeType','Assembly_Neurons_ID']).size().reset_index(name='count')
pivot_df = df_melted_.pivot(index='ExpeType', columns='Assembly_Neurons_ID', values='count').fillna(0)
size = 0.35
vals_outer = df_melted_[df_melted_['ExpeType']=='Cheeseboard']['count'].tolist()
vals_inner = df_melted_[df_melted_['ExpeType']=='SleepAfter']['count'].tolist()
color_map = ['blue', 'green', 'orange', 'black', 'grey', 'purple']

ax.pie(vals_outer, radius=1.2, colors=color_map, #labels=['Spatial', 'Non_Spatial', 'unclassified', 'mix'],
       wedgeprops=dict(width=size, edgecolor='w'))
ax.pie(vals_inner, radius=1-size, colors=color_map, 
       wedgeprops=dict(width=size, edgecolor='w'))
ax.set(aspect="equal", title= 'Cheeseboard \n SleepAfter')
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblyProportion_NestedPieChart_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

### Spatially-tuned vs non Spatially-tuned

Plot Reactivation of cell assemblies

In [ ]:
NewCellAss_ID_pivot= df_NewCellAss_ID.pivot_table(index='New_Assembly_ID', columns=['ExpeType'], values='EventFreq', aggfunc='mean', fill_value=None)
NewCellAss_ID_pivot_clean = NewCellAss_ID_pivot.dropna(subset=['Cheeseboard', 'SleepAfter'])
NewCellAss_ID_pivot_clean1 = pd.merge(NewCellAss_ID_pivot_clean,df_NewCellAss_ID[['Assembly_Spatial_ID', 'New_Assembly_ID']], on='New_Assembly_ID', how='inner')
NewCellAss_ID_pivot_clean1 = NewCellAss_ID_pivot_clean1.drop_duplicates()

plt.figure(figsize=(2,4))
color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'unclassified': 'grey', 'mix': 'purple'} 
timepoints = [1, 2]
for idx, row in NewCellAss_ID_pivot_clean1.iterrows():
    plt.plot(timepoints, [row['Cheeseboard'], row['SleepAfter']], color=color_map[row['Assembly_Spatial_ID']])
plt.xticks(timepoints, ['Cheeseboard', 'PostSession'])
plt.ylabel('Cell assembly events (Hz)')
plt.title(f'n = {len(NewCellAss_ID_pivot_clean1)}/{len(NewCellAss_ID_pivot)} cell assemblies')
plt.legend(handles=[plt.Line2D([0], [0], color=color, lw=2, label=label) for label, color in color_map.items()], title='Majority of', bbox_to_anchor=(1.05, 1), loc='upper left')
print(f'{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot)*100)} % of cell assemblies are detected during both Cheeseboard and SleepAfter sessions')
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['Cheeseboard']))*100)} % of cell assemblies detected during Cheeseboard are detected in SleepAfter sessions")
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['SleepAfter']))*100)} % of cell assemblies detected during SleepAfter were detected in Cheeseboard sessions")
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblyEvents-Spatial_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
df_NewCellAss_ID_ = df_NewCellAss_ID[~(df_NewCellAss_ID['ExpeType'] == 'SleepAfter') | (df_NewCellAss_ID['Substate'] == 'NREM')]
NewCellAss_ID_pivot= df_NewCellAss_ID_.pivot_table(index='New_Assembly_ID', columns=['Substate'], values='EventFreq', aggfunc='mean', fill_value=None)
NewCellAss_ID_pivot_clean = NewCellAss_ID_pivot.dropna(subset=['AW', 'NREM'])
NewCellAss_ID_pivot_clean1 = pd.merge(NewCellAss_ID_pivot_clean,df_NewCellAss_ID_[['Assembly_Spatial_ID', 'New_Assembly_ID']], on='New_Assembly_ID', how='inner')
NewCellAss_ID_pivot_clean1 = NewCellAss_ID_pivot_clean1.drop_duplicates()

plt.figure(figsize=(2,4))
color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'unclassified': 'grey', 'mix': 'purple'} 
timepoints = [1, 2]
for idx, row in NewCellAss_ID_pivot_clean1.iterrows():
    plt.plot(timepoints, [row['AW'], row['NREM']], color=color_map[row['Assembly_Spatial_ID']])
plt.xticks(timepoints, ['Cheeseboard', 'NREM PostSession'])
plt.ylabel('Cell assembly events (Hz)')
plt.title(f'n = {len(NewCellAss_ID_pivot_clean1)}/{len(NewCellAss_ID_pivot)} cell assemblies')
plt.legend(handles=[plt.Line2D([0], [0], color=color, lw=2, label=label) for label, color in color_map.items()], title='Majority of', bbox_to_anchor=(1.05, 1), loc='upper left')
print(f'{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot)*100)} % of cell assemblies are detected during both AW Cheeseboard and NREM SleepAfter sessions')
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['AW']))*100)} % of cell assemblies detected during AW Cheeseboard are detected in NREM SleepAfter sessions")
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['NREM']))*100)} % of cell assemblies detected during NREM SleepAfter were detected in AW Cheeseboard sessions")
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblyEvents_Spatial_inNREM.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
df_NewCellAss_ID_ = df_NewCellAss_ID[~(df_NewCellAss_ID['ExpeType'] == 'SleepAfter') | (df_NewCellAss_ID['Substate'] == 'REM')]
NewCellAss_ID_pivot= df_NewCellAss_ID_.pivot_table(index='New_Assembly_ID', columns=['Substate'], values='EventFreq', aggfunc='mean', fill_value=None)
NewCellAss_ID_pivot_clean = NewCellAss_ID_pivot.dropna(subset=['AW', 'REM'])
NewCellAss_ID_pivot_clean1 = pd.merge(NewCellAss_ID_pivot_clean,df_NewCellAss_ID_[['Assembly_Spatial_ID', 'New_Assembly_ID']], on='New_Assembly_ID', how='inner')
NewCellAss_ID_pivot_clean1 = NewCellAss_ID_pivot_clean1.drop_duplicates()

plt.figure(figsize=(2,4))
color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'unclassified': 'grey', 'mix': 'purple'} 
timepoints = [1, 2]
for idx, row in NewCellAss_ID_pivot_clean1.iterrows():
    plt.plot(timepoints, [row['AW'], row['REM']], color=color_map[row['Assembly_Spatial_ID']])
plt.xticks(timepoints, ['Cheeseboard', 'REM PostSession'])
plt.ylabel('Cell assembly events (Hz)')
plt.title(f'n = {len(NewCellAss_ID_pivot_clean1)}/{len(NewCellAss_ID_pivot)} cell assemblies')
plt.legend(handles=[plt.Line2D([0], [0], color=color, lw=2, label=label) for label, color in color_map.items()], title='Majority of', bbox_to_anchor=(1.05, 1), loc='upper left')
print(f'{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot)*100)} % of cell assemblies are detected during both AW Cheeseboard and REM SleepAfter sessions')
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['AW']))*100)} % of cell assemblies detected during AW Cheeseboard are detected in REM SleepAfter sessions")
print(f"{round(len(NewCellAss_ID_pivot_clean1)/len(NewCellAss_ID_pivot.dropna(subset=['REM']))*100)} % of cell assemblies detected during REM SleepAfter were detected in AW Cheeseboard sessions")
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblyEvents_Spatial_inREM.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Cell assembly size

In [ ]:
NewCellAss_ID_pivot = df_NewCellAss_ID.pivot_table(index='New_Assembly_ID', columns=['ExpeType'], values='Assembly_size', aggfunc='mean', fill_value=None)
NewCellAss_ID_pivot_clean = pd.merge(NewCellAss_ID_pivot,df_NewCellAss_ID[['Assembly_Spatial_ID', 'New_Assembly_ID']], on='New_Assembly_ID', how='inner')
NewCellAss_ID_pivot_clean = NewCellAss_ID_pivot_clean.drop_duplicates()

color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'unclassified': 'grey', 'mix': 'purple'} 

df_melted = NewCellAss_ID_pivot_clean.melt(
    id_vars='Assembly_Spatial_ID',
    value_vars=['Cheeseboard', 'SleepAfter'],
    var_name='ExpeType',
    value_name='Assembly_size'
)
df_melted['Assembly_Spatial_ID'] = pd.Categorical(df_melted['Assembly_Spatial_ID'], ["Spatial", "Non_Spatial", "unclassified", "mix"])

fig, ax= plt.subplots(figsize=(13,2.5))
sns.swarmplot(
    x='ExpeType', 
    y='Assembly_size', 
    data=df_melted, 
    hue='Assembly_Spatial_ID', 
    palette=color_map, 
    size=6, 
    alpha=1,
    ax= ax,
)
plt.ylabel('Cell Assembly size')
plt.xticks([0,1], ['Cheeseboard', 'SleepAfter'])
plt.legend(loc='upper right', title='Majority of')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblySize_Spatial_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
print(f"{df_NewCellAss_ID[df_NewCellAss_ID['ExpeType']=='Cheeseboard']['New_Assembly_ID'].nunique()} cell assemblies detected during Cheeseboard sessions")
print(f"{df_NewCellAss_ID[df_NewCellAss_ID['ExpeType']=='SleepAfter']['New_Assembly_ID'].nunique()} cell assemblies detected during SleepAfter sessions")
plt.show()


In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(7, 3), sharey=False)
color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'unclassified': 'grey', 'mix': 'purple'} 

df_melted_ = df_melted.dropna().groupby(['ExpeType','Assembly_Spatial_ID']).size().reset_index(name='count')
pivot_df = df_melted_.pivot(index='ExpeType', columns='Assembly_Spatial_ID', values='count').fillna(0)
pivot_df.plot(kind='bar', stacked=True, color=[color_map[col] for col in pivot_df.columns], ax= ax1)
ax1.set_ylabel('# of cell assemblies')
ax1.set_xlabel('')
ax1.legend(title='Majority of', bbox_to_anchor=(1.05, 1), loc='upper left')
for label in ax1.get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')

Sum = pivot_df[['Spatial', 'Non_Spatial', 'unclassified', 'mix']].sum(axis=1)
pivot_df[['Spatial', 'Non_Spatial', 'unclassified', 'mix']] = pivot_df[['Spatial', 'Non_Spatial', 'unclassified', 'mix']].div(Sum, axis=0)*100
pivot_df.plot(kind='bar', stacked=True, color=[color_map[col] for col in pivot_df.columns], ax= ax2, legend=False)
ax2.set_ylabel('Cell assemblies (%)')
ax2.set_xlabel('')
for label in ax2.get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')

pivot_= NewCellAss_ID_pivot_clean.dropna().groupby(['Assembly_Spatial_ID']).size().reset_index(name='count')
Sum = pivot_['count'].sum(axis=0)
pivot_['count'] = pivot_['count'].div(Sum, axis=0)*100
pivot_ = pivot_.set_index('Assembly_Spatial_ID').loc[['Spatial', 'Non_Spatial', 'mix']].reset_index()
bottom = 0
for color, value in zip(pivot_['Assembly_Spatial_ID'], pivot_['count']):
    ax3.bar(['Both sessions'], value, bottom=bottom, color=color_map[color])
    bottom += value
ax3.set_ylabel('Cell assemblies (%)')
for label in ax3.get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')

plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblyProportion_Spatial_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(2.5,2.5))
df_melted_ = df_melted.dropna().groupby(['ExpeType','Assembly_Spatial_ID']).size().reset_index(name='count')
pivot_df = df_melted_.pivot(index='ExpeType', columns='Assembly_Spatial_ID', values='count').fillna(0)
size = 0.35
vals_outer = df_melted_[df_melted_['ExpeType']=='Cheeseboard']['count'].tolist()
vals_inner = df_melted_[df_melted_['ExpeType']=='SleepAfter']['count'].tolist()
color_map = ['blue', 'orange', 'grey','purple',]

ax.pie(vals_outer, radius=1.2, colors=color_map, #labels=['Spatial', 'Non_Spatial', 'unclassified', 'mix'],
       wedgeprops=dict(width=size, edgecolor='w'))
ax.pie(vals_inner, radius=1-size, colors=color_map, 
       wedgeprops=dict(width=size, edgecolor='w'))
ax.set(aspect="equal", title= 'Cheeseboard \n SleepAfter')
plt.savefig(f'{Path(dfilepath).parent}/NewCellAssemblyProportion_Spatial_NestedPieChart_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

## ClusterHDBSCAN & Cell assembly 

### Cell assembly identity

Load VigSt_Global_cluster file

In [ ]:
#with open(f'{Path(dfilepath).parent}/VigStates_Global_cluster.pkl', 'rb') as pickle_file:
with open("//10.69.168.1/crnldata/waking/audrey_hay/L1imaging/Analysed2025_AB/_CGP_analysis/VigSt_2025-05-21_15_47_42_CGP/VigStates_Global_cluster.pkl", 'rb') as pickle_file:
    df_cluster = pickle.load(pickle_file)

df_cluster = df_cluster[df_cluster['NeuronType'] == 'L1NDNF_mice']
df_cluster_Drug = df_cluster.copy()

#df_cluster_Drug = df_cluster_Drug[df_cluster_Drug['Drug'] == 'baseline']

AllBaselineUnits = df_cluster_Drug['Unit_ID'].unique()
Cluster0units = df_cluster_Drug[df_cluster_Drug['ClusterHDBSCAN'] == 0]['Unit_ID'].unique()
Cluster1units = df_cluster_Drug[df_cluster_Drug['ClusterHDBSCAN'] == 1]['Unit_ID'].unique()
Cluster2units = df_cluster_Drug[df_cluster_Drug['ClusterHDBSCAN'] == 2]['Unit_ID'].unique()

In [ ]:
# --- Setup: Individuals and clusters
new_df2 = combined_df.drop_duplicates(subset='Assembly_ID', keep='first')
new_df2 = new_df2[new_df2['Cells_in_Assembly'].apply(len) == new_df2['Assembly_size']]
groups = new_df2['Cells_in_Assembly'].tolist()
df_unique = df_cluster_Drug.drop_duplicates(subset='Unit_ID', keep='first')
df_unique = df_unique.dropna(subset=['ClusterHDBSCAN'])
ids=df_unique['Unit_ID'].tolist()
id_to_cluster = dict(zip(df_unique['Unit_ID'], np.floor(df_unique['ClusterHDBSCAN']).astype(str)))
cluster_labels = ['0.0']*len(Cluster0units) + ['1.0']*len(Cluster1units) + ['2.0']*len(Cluster2units)
#cluster_labels = new_df2_c['Assembly_Cluster_ID'].tolist()

Identify Cell assembly Cluster ID and test proportion

In [ ]:
# --- Setup: Individuals and clusters
new_df2 = combined_df.drop_duplicates(subset='Assembly_ID', keep='first')
new_df2 = new_df2[new_df2['Cells_in_Assembly'].apply(len) == new_df2['Assembly_size']]
groups = new_df2['Cells_in_Assembly'].tolist()
df_unique = df_cluster_Drug.drop_duplicates(subset='Unit_ID', keep='first')
df_unique = df_unique.dropna(subset=['ClusterHDBSCAN'])
ids=df_unique['Unit_ID'].tolist()
id_to_cluster = dict(zip(df_unique['Unit_ID'], np.floor(df_unique['ClusterHDBSCAN']).astype(str)))
cluster_labels = ['0.0']*len(Cluster0units) + ['1.0']*len(Cluster1units) + ['2.0']*len(Cluster2units)
#cluster_labels = new_df2_c['Assembly_Cluster_ID'].tolist()

def classify_groups(groups, id_to_cluster):
    labels = []
    for group in groups:
        clusters = [id_to_cluster.get(i, np.nan) for i in group]
        if len(clusters) >= 1:
            count = Counter(clusters)
            top, n_top = count.most_common(1)[0]
            top_clusters = [k for k, v in count.items() if v == n_top]        
            # If there are multiple top clusters or the majority is less than 50%, label as "mix"
            if len(top_clusters) == 1 and (n_top / len(group) * 100) > 50 and not pd.isna(top):
                labels.append(top)
            else:
                labels.append("mix")  
        else:
            labels.append("mix")    
    return labels

# --- Observed classification
observed_labels = classify_groups(groups, id_to_cluster)
observed_counts = Counter(observed_labels)
new_df2['Assembly_Cluster_ID']= observed_labels

# --- Permutation test
n_perms = 5000
null_dists = defaultdict(list)

for _ in range(n_perms):
    # Shuffle cluster labels across individuals
    shuffled_clusters = dict(zip(ids, random.sample(cluster_labels, len(cluster_labels))))
    shuffled_labels = classify_groups(groups, shuffled_clusters)
    counts = Counter(shuffled_labels)
    for label in observed_counts:
        null_dists[label].append(counts.get(label, 0))

# --- Z-scores and p-values
results = []
for label in observed_counts:
    obs = observed_counts[label]
    null = null_dists[label]
    mean = np.mean(null)
    std = np.std(null)
    z = (obs - mean) / std if std > 0 else 0
    # Two-tailed p-value
    p = (np.sum(np.abs(np.array(null) - mean) >= abs(obs - mean)) + 1) / (n_perms + 1)
    results.append((label, obs, mean, std, z, p))

# --- Output
print("\nAssembly Cluster ID Distribution Test")
print(f"{'ID':>10} {'Obs':>5} {'Mean':>7} {'Std':>7} {'Z':>6} {'p-value':>8}")
for label, obs, mean, std, z, p in sorted(results, key=lambda x: x[-1]):
    print(f"{label:>10} {obs:5} {mean:7.2f} {std:7.2f} {z:6.2f} {p:8.4f}")


df = pd.DataFrame(results, columns=["Cell_Assembly_ID", "Obs", "PermMean", "Std", "Z", "p"])
df['Obs_prop'] = df['Obs'] / df['Obs'].sum() *100
df['Perm_prop'] = df['PermMean'] / df['PermMean'].sum() *100
df["Perm_std"] = df["Std"] / df["PermMean"].sum() *100

df = df.sort_values("Cell_Assembly_ID")  # or use a custom order
plt.figure(figsize=(3, 3))
plt.errorbar(df["Cell_Assembly_ID"], df["Perm_prop"], yerr=df["Perm_std"], fmt='o', color='gray', label='Permuted ± std')
plt.scatter(df["Cell_Assembly_ID"], df["Obs_prop"], color='black', label='Observed')
plt.ylabel("Proportion")
plt.ylim(0, 100)
plt.title("Observed vs Permuted Proportions")
plt.legend()
plt.tight_layout()
#plt.savefig(f'C:/Users/Manip2/Documents/Manuscripts/Figures_PNASreviews/Figure2_revised/CellAssembly_Proportion_permvsreal_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Save cell assembly ID 

In [ ]:
merged = pd.merge(combined_df0, new_df2[['Assembly_ID', 'Assembly_Cluster_ID']], on='Assembly_ID', how='outer')
merged.to_excel(f'{Path(dfilepath).parent}/CellAssembly_Global_cluster_ID.xlsx')

Activity of cell assemblies per ID

In [ ]:
merged = merged[merged['Substate'] != 'IS']
merged = merged[merged['Substate'] != 'undefined']
combined_df2 = merged.pivot_table(index='Assembly_ID', columns=[merged['Assembly_Cluster_ID'], merged['Substate']], values='Avg_Activity', aggfunc='mean', fill_value=None)
custom_order = ['AW', 'QW', 'NREM', 'REM']
sort_key = {val: i for i, val in enumerate(custom_order)}
sorted_cols = sorted(combined_df2.columns, key=lambda x: (x[0], sort_key.get(x[1], float('inf'))))
combined_df2 = combined_df2[sorted_cols]

plt.figure(figsize=(7, 3))
for idx, row in combined_df2.iterrows():
    plt.plot(row.values, label=f'{idx}', alpha=0.5, linewidth=2)  # `idx` is the MultiIndex for each row
plt.ylabel('Avg activity per cell assemblies')
plt.tight_layout()
plt.xticks(ticks= np.arange(len(combined_df2.columns.get_level_values(1))),  labels=combined_df2.columns.get_level_values(1)) 
plt.title('Cluster0                Cluster1                Cluster2                   mix')


combined_df2 = merged.pivot_table(index='Assembly_ID', columns=[merged['Assembly_Cluster_ID'], merged['Substate']], values='EventFreq', aggfunc='mean', fill_value=None)
custom_order = ['AW', 'QW', 'NREM', 'REM']
sort_key = {val: i for i, val in enumerate(custom_order)}
sorted_cols = sorted(combined_df2.columns, key=lambda x: (x[0], sort_key.get(x[1], float('inf'))))
combined_df2 = combined_df2[sorted_cols]

plt.figure(figsize=(7, 3))
for idx, row in combined_df2.iterrows():
    plt.plot(row.values, label=f'{idx}', alpha=0.5, linewidth=2)  # `idx` is the MultiIndex for each row
plt.ylabel('Event frequency per cell assemblies')
plt.tight_layout()
plt.xticks(ticks= np.arange(len(combined_df2.columns.get_level_values(1))),  labels=combined_df2.columns.get_level_values(1)) 
plt.title('Cluster0                Cluster1                Cluster2                   mix')

plt.show()

In [ ]:
plt.savefig(f"C:/Users/Manip2/Documents/Manuscripts/Figures_PNASreviews/Figure2_revised/{NrSubtype}_.svg", format='svg')

### Cell assemblies activity around oscillation

In [ ]:
dir = "//10.69.168.1/crnldata/waking/audrey_hay/L1imaging/Analysed2025_AB/"
DrugExperiment=0
all_expe_types=['postCGP'] if DrugExperiment else ['baseline', 'preCGP']
AHmethod=0

CA_PFC_L2_3_mice=pd.DataFrame()
CA_PFC_L2_3_mice_n=pd.DataFrame()
CA_PFC_L1NDNF_mice=pd.DataFrame()
CA_PFC_L1NDNF_mice_n=pd.DataFrame()
CA_PFC_L1NDNF_mice_m=pd.DataFrame()
CA_PFC_L1NDNF_mice_0=pd.DataFrame()
CA_PFC_L1NDNF_mice_1=pd.DataFrame()
CA_PFC_L1NDNF_mice_2=pd.DataFrame()

CA_S1_L2_3_mice=pd.DataFrame()
CA_S1_L2_3_mice_n=pd.DataFrame()
CA_S1_L1NDNF_mice=pd.DataFrame()
CA_S1_L1NDNF_mice_n=pd.DataFrame()
CA_S1_L1NDNF_mice_m=pd.DataFrame()
CA_S1_L1NDNF_mice_0=pd.DataFrame()
CA_S1_L1NDNF_mice_1=pd.DataFrame()
CA_S1_L1NDNF_mice_2=pd.DataFrame()

CA_S1PFC_L2_3_mice=pd.DataFrame()
CA_S1PFC_L2_3_mice_n=pd.DataFrame()
CA_S1PFC_L1NDNF_mice=pd.DataFrame()
CA_S1PFC_L1NDNF_mice_n=pd.DataFrame()
CA_S1PFC_L1NDNF_mice_m=pd.DataFrame()
CA_S1PFC_L1NDNF_mice_0=pd.DataFrame()
CA_S1PFC_L1NDNF_mice_1=pd.DataFrame()
CA_S1PFC_L1NDNF_mice_2=pd.DataFrame()

CA_SWR_L2_3_mice=pd.DataFrame()
CA_SWR_L2_3_mice_n=pd.DataFrame()
CA_SWR_L1NDNF_mice=pd.DataFrame()
CA_SWR_L1NDNF_mice_n=pd.DataFrame()
CA_SWR_L1NDNF_mice_m=pd.DataFrame()
CA_SWR_L1NDNF_mice_0=pd.DataFrame()
CA_SWR_L1NDNF_mice_1=pd.DataFrame()
CA_SWR_L1NDNF_mice_2=pd.DataFrame()

window = [-5, 5]  # from -10s to +10s around each event
bin_width = 0.1
bins = np.arange(window[0], window[1] + bin_width, bin_width)

dfilepath="//10.69.168.1/crnldata/waking/audrey_hay/L1imaging/Analysed2025_AB/_CGP_analysis/VigSt_2025-05-21_15_47_42_CGP/CellAssembly_Global_cluster_ID.xlsx"
try :
    cellassemblydf0 = pd.read_excel(dfilepath, index_col=0)
except:
    with open(dfilepath, 'rb') as pickle_file:
        cellassemblydf0 = pickle.load(pickle_file)

for dpath in Path(dir).glob('**/mappingsAB.pkl'):
    
    mappfile = open(dpath.parents[0]/ f'mappingsAB.pkl', 'rb')
    mapping = pickle.load(mappfile)
    mapping_sess = mapping['session']   

    mice = dpath.parents[0].parts[-1]
    NeuronType = dpath.parents[1].parts[-1]
    
    subsessions = []
    dict_S1PFCprop = {}
    dict_Spindleprop = {}
    dict_Path={}

    minian_folders = [f for f in dpath.parents[0].rglob('minian') if f.is_dir()]

    for minianpath in minian_folders: # for each minian folders found in this mouse

        if any(p in all_expe_types for p in minianpath.parts): # have to be to the expe_types
                        
            session=minianpath.parents[0].name if len(minianpath.parts)==12 else minianpath.parents[1].name.split("_")[-1]
            session_path=minianpath.parents[2] if len(minianpath.parts)==12 else minianpath.parents[1]
            expe_type=minianpath.parents[3].name if len(minianpath.parts)==12 else minianpath.parents[2].name
            
            session_date= minianpath.parents[2].name.split("_")[0] if len(minianpath.parts)==12 else minianpath.parents[1].name.split("_")[0]
            session_time= minianpath.parents[2].name.split("_")[1] if len(minianpath.parts)==12 else minianpath.parents[1].name.split("_")[1]

            drug='CGP' if expe_type == 'postCGP' else 'baseline'
            dict_Path[session] = session_path
            
            SWRlist= pd.read_csv(session_path / f'OpenEphys/SWRproperties.csv' ) if AHmethod else pd.read_csv(session_path / f'OpenEphys/SWR_finedetection.csv' ) 
            SWRlist['toKeep'] = 'True' # SWRlist['toKeep'].astype(str)
            SWRprop = SWRlist[SWRlist['toKeep'].isin(['VRAI', 'True'])]
            
            startSWRlist=SWRprop["start time"]/1000
            merged = cellassemblydf0.groupby(['Mice', 'Session_Date','Session_Time', 'Assembly_ID', 'Assembly_Cluster_ID'], dropna=False)['EventTime'].apply(lambda x: ', '.join(x)).reset_index() 
            for ass in merged.loc[(merged['Mice'] == mice) & (merged['Session_Date'] == session_date) & (merged['Session_Time'] == session_time)].index:
                CA_SWR_matrix=locals()[f"CA_SWR_{NeuronType}_{str(merged['Assembly_Cluster_ID'].loc[ass])[0]}"]
                CA_SWR_matrix2=locals()[f"CA_SWR_{NeuronType}"]
                COUNT=[]
                numbers = re.findall(r'[\d.]+',  merged['EventTime'].loc[ass])
                events = np.array(numbers, dtype=float)
                if len(events)>0:
                    for SWRstart in startSWRlist:                        
                        relativedistance= SWRstart-events
                        counts, _ = np.histogram(relativedistance, bins=bins)                  
                        COUNT.append(counts)                               
                    CA_SWR_matrix[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.nanmean(COUNT, axis=0)
                    CA_SWR_matrix2[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.nanmean(COUNT, axis=0)
                else: 
                    CA_SWR_matrix[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.zeros(len(bins)-1)
                    CA_SWR_matrix2[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.zeros(len(bins)-1)
                

            
            Spdllist = pd.read_csv(session_path / f'OpenEphys/Spindleproperties_S1&PFC.csv') if AHmethod else pd.read_csv(session_path / f'OpenEphys/SpindlesS1&PFC_finedetection.csv' ) 
            Spdllist['toKeep'] = 'True' # Spdllist['toKeep'].astype(str)
            Spdlprop  = Spdllist[Spdllist['toKeep'].isin(['VRAI', 'True'])]

            S1Spdlprop= Spdlprop[Spdlprop['CTX']=='S1']
            S1PFCSpdlprop= Spdlprop[Spdlprop['CTX']=='S1PFC']
            PFCSpdlprop= Spdlprop[Spdlprop['CTX']=='PFC']
            
            startPFClist=PFCSpdlprop["start time"]/1000
            merged = cellassemblydf0.groupby(['Mice', 'Session_Date','Session_Time', 'Assembly_ID', 'Assembly_Cluster_ID'], dropna=False)['EventTime'].apply(lambda x: ', '.join(x)).reset_index() 
            for ass in merged.loc[(merged['Mice'] == mice) & (merged['Session_Date'] == session_date) & (merged['Session_Time'] == session_time)].index:
                CA_PFC_matrix=locals()[f"CA_PFC_{NeuronType}_{str(merged['Assembly_Cluster_ID'].loc[ass])[0]}"]
                CA_PFC_matrix2=locals()[f"CA_PFC_{NeuronType}"]
                COUNT=[]
                numbers = re.findall(r'[\d.]+',  merged['EventTime'].loc[ass])
                events = np.array(numbers, dtype=float)
                if len(events)>0:
                    for spdlstart in startPFClist:                        
                        relativedistance= spdlstart-events
                        counts, _ = np.histogram(relativedistance, bins=bins)                  
                        COUNT.append(counts)                               
                    CA_PFC_matrix[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.nanmean(COUNT, axis=0)
                    CA_PFC_matrix2[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.nanmean(COUNT, axis=0)
                else: 
                    CA_PFC_matrix[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.zeros(len(bins)-1)
                    CA_PFC_matrix2[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.zeros(len(bins)-1)
                
            
            startS1list=S1Spdlprop["start time"]/1000
            merged = cellassemblydf0.groupby(['Mice', 'Session_Date','Session_Time', 'Assembly_ID', 'Assembly_Cluster_ID'], dropna=False)['EventTime'].apply(lambda x: ', '.join(x)).reset_index() 
            for ass in merged.loc[(merged['Mice'] == mice) & (merged['Session_Date'] == session_date) & (merged['Session_Time'] == session_time)].index:
                CA_S1_matrix=locals()[f"CA_S1_{NeuronType}_{str(merged['Assembly_Cluster_ID'].loc[ass])[0]}"]
                CA_S1_matrix2=locals()[f"CA_S1_{NeuronType}"]
                COUNT=[]
                numbers = re.findall(r'[\d.]+',  merged['EventTime'].loc[ass])
                events = np.array(numbers, dtype=float)
                if len(events)>0:
                    for spdlstart in startS1list:                        
                        relativedistance= spdlstart-events
                        counts, _ = np.histogram(relativedistance, bins=bins)                  
                        COUNT.append(counts)                               
                    CA_S1_matrix[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.nanmean(COUNT, axis=0)
                    CA_S1_matrix2[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.nanmean(COUNT, axis=0)
                else: 
                    CA_S1_matrix[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.zeros(len(bins)-1)
                    CA_S1_matrix2[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.zeros(len(bins)-1)

            
            startS1PFClist=S1PFCSpdlprop["start time"]/1000
            merged = cellassemblydf0.groupby(['Mice', 'Session_Date','Session_Time', 'Assembly_ID', 'Assembly_Cluster_ID'], dropna=False)['EventTime'].apply(lambda x: ', '.join(x)).reset_index() 
            for ass in merged.loc[(merged['Mice'] == mice) & (merged['Session_Date'] == session_date) & (merged['Session_Time'] == session_time)].index:
                CA_S1PFC_matrix=locals()[f"CA_S1PFC_{NeuronType}_{str(merged['Assembly_Cluster_ID'].loc[ass])[0]}"]
                CA_S1PFC_matrix2=locals()[f"CA_S1PFC_{NeuronType}"]
                COUNT=[]
                numbers = re.findall(r'[\d.]+',  merged['EventTime'].loc[ass])
                events = np.array(numbers, dtype=float)
                if len(events)>0:
                    for spdlstart in startS1PFClist:                        
                        relativedistance= spdlstart-events
                        counts, _ = np.histogram(relativedistance, bins=bins)                  
                        COUNT.append(counts)                               
                    CA_S1PFC_matrix[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.nanmean(COUNT, axis=0)
                    CA_S1PFC_matrix2[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.nanmean(COUNT, axis=0)
                else: 
                    CA_S1PFC_matrix[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.zeros(len(bins)-1)
                    CA_S1PFC_matrix2[merged['Assembly_ID'].loc[ass]+ '_' + str(merged['Assembly_Cluster_ID'].loc[ass])[0]]=np.zeros(len(bins)-1)

In [ ]:
CA_matrix_all = ['CA_S1PFC_L2_3_mice', 'CA_S1PFC_L1NDNF_mice', 'CA_S1_L2_3_mice', 'CA_S1_L1NDNF_mice', 'CA_PFC_L2_3_mice', 'CA_PFC_L1NDNF_mice', 'CA_SWR_L2_3_mice', 'CA_SWR_L1NDNF_mice']#, 'CA_S1PFC_L1NDNF_mice_n', 'CA_S1PFC_L1NDNF_mice_m',                      'CA_S1PFC_L1NDNF_mice_0', 'CA_S1PFC_L1NDNF_mice_1', 'CA_S1PFC_L1NDNF_mice_2']
label_colors = {    'n': 'white',    'm': 'grey',    '0': 'blue',     '1': 'cyan',     '2': 'green'}
drug='CGP' if DrugExperiment else 'baseline'

plt.close()
for CA_SWR_matrix_name in CA_matrix_all: 
    CA_SWR_matrix= locals()[CA_SWR_matrix_name]
    CA_SWR_matrix_n = CA_SWR_matrix/ CA_SWR_matrix.max() 
    CA_SWR_matrix_n = CA_SWR_matrix_n.replace(np.nan, 0)
    max_row_pos = CA_SWR_matrix_n.values.argmax(axis=0)
    col_sums = CA_SWR_matrix_n.sum()
    col_info = pd.DataFrame({'col': CA_SWR_matrix_n.columns,'max_pos': max_row_pos,'sum': col_sums.values})
    col_info_sorted = col_info.sort_values(by=['max_pos', 'sum'])
    CA_SWR_matrix_s = CA_SWR_matrix_n[col_info_sorted['col'].values]
    CA_SWR_matrix_s = CA_SWR_matrix_s.loc[:, (CA_SWR_matrix_s != 0).any(axis=0)]
    df= CA_SWR_matrix_s.T

    # Row label to color index mapping
    row_labels = df.index.str[-1]
    color_codes = [list(label_colors).index(lbl) for lbl in row_labels]
    color_array = np.array(color_codes).reshape(-1, 1)
    cmap = ListedColormap([label_colors[k] for k in label_colors])
    fig = plt.figure(figsize=(3, 3))
    gs = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[3, 2],   width_ratios=[0.5, 10], hspace=0.3, wspace=0.05 )
    ax_ann = fig.add_subplot(gs[0, 0])
    sns.heatmap(color_array, ax=ax_ann, cmap=cmap, cbar=False)
    ax_ann.set_xticks([])
    ax_ann.set_yticks([])
    ax_ann.tick_params(left=False, right=False)
    ax_heatmap = fig.add_subplot(gs[0, 1])
    sns.heatmap(df, ax=ax_heatmap, cmap="viridis", cbar=False, cbar_kws={'location': 'top'}, yticklabels=False)
    ax_heatmap.set_yticklabels([]) 
    ax_heatmap.set_title(f"n = {len(df)}/{len(CA_SWR_matrix_n.T)}") 
    ax_ann.set_ylabel(CA_SWR_matrix_name)

    time_bins = np.linspace(window[0], window[1], len(bins)-1)
    ax_bottom = fig.add_subplot(gs[1, 1])  # span both columns
    ax_bottom.plot(time_bins, zscore(np.mean(df, axis=0)))
    ax_bottom.set_xlabel(f"Time from {CA_SWR_matrix_name.split('_')[1]} (s)")
    ax_bottom.set_xticks([])
    ax_bottom.set_xlim(window[0], window[1])
    ax_bottom.set_ylim(-2.5, 5)

    # Monte Carlo parameters
    n_permutations = 5000
    shuffled_psths = []
    for _ in range(n_permutations):
        shuffled_counts = np.apply_along_axis(np.random.permutation, 1, df)
        shuffled_psth = shuffled_counts.mean(axis=0)
        shuffled_psths.append(shuffled_psth)
    shuffled_psths = np.array(shuffled_psths)
    alpha = 0.05
    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_bins05 = (np.mean(df, axis=0) < lower_bound) | (np.mean(df, axis=0) > upper_bound)
    ax_bottom.scatter(time_bins[significant_bins05], np.linspace(4, 4, len(time_bins))[significant_bins05], marker='_', s=5, color= 'r')
    plt.savefig(f"C:/Users/Manip2/Documents/Manuscripts/Figures_PNASreviews/Figure4_revised/CellAssembly_Heatmap_{CA_SWR_matrix_name.split('_')[2]}_{CA_SWR_matrix_name.split('_')[1]}_{drug}.svg", format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
    plt.show()